# 06 · Mobility and shared-setting exposure

**Question:** How can an event log connect daily activity to a toy exposure model?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
All city geometry, residents, sensor observations and parameters are **synthetic educational fixtures**.
The longitude/latitude anchor is near Gaborone, but these are not mapped Gaborone assets or a validated city twin.

**Astra experiment:** Have Astra audit the event schema, explain the effect of ventilation, and identify what is missing from a TB natural-history model.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## From places to events
160 synthetic agents occupy homes, workplaces and shared vehicles for one day.
Four fixed source agents contribute an arbitrary dose. Dose is not infection probability, TB incidence, or a calibrated clinical quantity.
This intentionally small model is not Starsim and does not reproduce Kopanyo findings.


In [ ]:
ventilation_multiplier=2.0
baseline=mobility(ventilation=1.)
scenario=mobility(ventilation=ventilation_multiplier)
sus=baseline['susceptible']
assert baseline['trajectory'].shape==(48,160)
assert np.array_equal(baseline['trajectory'],scenario['trajectory'])
assert np.allclose(scenario['dose'],baseline['dose']/ventilation_multiplier)
print('Shared-setting event rows:',len(baseline['events']))
print('Mean dose among non-source agents:',baseline['dose'][sus].mean(),'->',scenario['dose'][sus].mean())


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].imshow(baseline['trajectory'][:,:30].T,aspect='auto',origin='lower',extent=[0,24,0,30],cmap='tab20')
axes[0].set(xlabel='Hour',ylabel='Agent',title='First 30 activity histories (place IDs)')
axes[1].hist([baseline['dose'][sus],scenario['dose'][sus]],bins=15,label=['Baseline','More ventilation'])
axes[1].set(xlabel='Arbitrary dose',ylabel='Agents',title='Shared trajectories; changed exposure'); axes[1].legend()
plt.tight_layout(); plt.show()
export_json('06_events.json',dict(synthetic=True,events=baseline['events'],ventilation_multiplier=ventilation_multiplier))


## Teaching lab: Review an event contract
**Before using AI:** write a prediction, run the numerical baseline, and explain one surprising result.

**Astra evidence:** send only 06_events.json and the arbitrary-dose assumptions. Use a fresh conversation and record the exact model,
date, interface, tools and prompt. A task with a supplied result tests interpretation, not independent calculation.

**Copyable Astra prompt:**
> Identify which fields establish shared occupancy and which information would still be needed for a disease model. Do not interpret dose as infection probability.

**Copyable Codex task:**
> Compare ventilation multipliers 1, 2 and 4 using the same trajectories, then repeat with three fixed seeds. Edit the authoring source in `scripts/make_notebooks.py` (or
> `scripts/teaching_labs.py` for labs 11–13), regenerate the notebooks and preserve browser compatibility.
> Use NumPy, Matplotlib and the standard library. Include a small reference case that can be checked by hand.

**Acceptance evidence:** Within each seed, trajectories are identical across interventions and dose scales inversely with ventilation.

**Share back:** save your original prediction, changed parameter, result, model response and one limitation.
Download the edited notebook and exports. If you have no model access, exchange explanations with a partner
and evaluate them with the same criteria; do not label that activity as an Astra trial.


## Extend with Codex or Astra
Connect agents to the building and street contracts. Add household/vehicle contact layers before porting a validated disease kernel. Use common random numbers and multiple seeds for scenario comparisons.

Keep the baseline seed and tests fixed while changing one assumption. Record the model name,
date, exact prompt, response and error metrics. Generated code must pass the same checks.
Download files in `exports/` and your edited notebook; browser storage does not commit changes to GitHub.
See [the project guide](https://github.com/jltobias/JupyterLite-GPT-6-Astra-Geospatial/blob/main/docs/GUIDE.md)
for capabilities, evidence, sources and the route to real data.
